# Data

## 1. Paths and prerequisites

We keep the raw competition files unchanged and prepare small metadata tables for EDA.
This notebook uses CPU metadata parsing and opens only a sample of image headers.

Run `uv sync --locked` first and select the project kernel. The first download is
about 10 GB. Local files take priority. Set `HM2_DATA_DIR` in `.env` to use an existing
directory or ZIP. Otherwise, [KaggleHub](https://github.com/Kaggle/kagglehub) reuses its
cache. Accept the competition rules in the browser and configure Kaggle authentication
or `KAGGLE_API_TOKEN` in `.env` before downloading. Credentials are never displayed.

The expected inventory comes from [the assignment context](../docs/assignment_context.md).

In [1]:
import json
import logging
import sys
from pathlib import Path
from zipfile import ZipFile

import kagglehub
import polars as pl
from decouple import config
from IPython.display import Markdown, display
from kagglehub.config import set_kaggle_api_token
from kagglehub.exceptions import KaggleApiHTTPError, UnauthenticatedError
from PIL import Image
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)

working_directory = Path.cwd().resolve()

REPOSITORY_ROOT = next(
    (
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "pyproject.toml").is_file() and (directory / "src/hm2").is_dir()
    )
)

METADATA_DIRECTORY = REPOSITORY_ROOT / "data/metadata"

sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

SEED = 2026

pl.Config.set_tbl_rows(12)

pl.Config.set_tbl_cols(12)

from hm2.data import (
    CLASS_NAMES,
    COMPETITION,
    audit_dataset,
    compare_yolo_coco,
    extract_archive,
    find_dataset_root,
    load_tables,
)

## 2. Locate or download the data

We check for extracted metadata before considering an archive or a download.
Extraction preserves wrapper directories and refuses to overwrite existing files.
An incomplete local dataset is reported by the checks below rather than overwritten.

In [2]:
configured_directory = config("HM2_DATA_DIR", default=None)

DATA_DIRECTORY = Path(configured_directory or REPOSITORY_ROOT / "data").expanduser()

dataset_root = find_dataset_root(DATA_DIRECTORY)

archives = [DATA_DIRECTORY] if DATA_DIRECTORY.is_file() else sorted(DATA_DIRECTORY.glob("*.zip"))

manifest_path = METADATA_DIRECTORY / "manifest.json"

if dataset_root is None and (not configured_directory) and manifest_path.is_file():
    previous_root = Path(json.loads(manifest_path.read_text())["dataset_root"])
    dataset_root = find_dataset_root(previous_root)

if dataset_root is None and archives:
    if len(archives) != 1:
        raise ValueError("More than one ZIP found. Set HM2_DATA_DIR to the intended ZIP.")
    with ZipFile(archives[0]) as archive:
        archive_summary = pl.DataFrame(
            {
                "Archive": [archives[0].name],
                "Files": [len(archive.infolist())],
                "Uncompressed GiB": [
                    sum((item.file_size for item in archive.infolist())) / 1024**3
                ],
            }
        )
        display(archive_summary)
        display(pl.DataFrame({"Example path": archive.namelist()[:8]}))
    extraction_directory = archives[0].parent / "extracted" / archives[0].stem
    if find_dataset_root(extraction_directory) is None:
        extract_archive(archives[0], extraction_directory)
    dataset_root = find_dataset_root(extraction_directory)

if dataset_root is None:
    token = config("KAGGLE_API_TOKEN", default=None)
    if token:
        set_kaggle_api_token(token)
    del token
    try:
        downloaded_path = Path(kagglehub.competition_download(COMPETITION))
    except KaggleApiHTTPError, UnauthenticatedError:
        raise RuntimeError(
            "Kaggle access failed. Check KAGGLE_API_TOKEN in .env or your existing Kaggle authentication, accept the competition rules, or set HM2_DATA_DIR to an existing local dataset."
        ) from None
    dataset_root = find_dataset_root(downloaded_path)

if dataset_root is None:
    raise FileNotFoundError("No test_images.csv found. Inspect the extracted archive layout.")

dataset_root = dataset_root.resolve()

pl.DataFrame({"Competition": [COMPETITION], "Dataset root": [str(dataset_root)]})

Competition,Dataset root
str,str
"""multiple-object-video-detectio…","""/Users/nikitalenyk/.cache/kagg…"


## 3. Dataset structure and inventory

We read train and validation COCO metadata and the test CSV. Target categories are
checked against the eight-class mapping. COCO crowd annotations remain in a separate
ignored-region table. A test frame has no target count because its labels are unknown.

In [3]:
layout = []

for split in ("train", "val", "test"):
    for directory in (dataset_root / split).iterdir():
        layout.append(
            {
                "Split": split,
                "Path": str(directory.relative_to(dataset_root)),
                "Kind": "Directory" if directory.is_dir() else "File",
            }
        )

pl.from_dicts(layout)

Split,Path,Kind
str,str,str
"""train""","""train/images""","""Directory"""
"""train""","""train/ignore""","""Directory"""
"""train""","""train/labels""","""Directory"""
"""train""","""train/annotations_coco.json""","""File"""
"""train""","""train/data.yaml""","""File"""
"""val""","""val/images""","""Directory"""
"""val""","""val/ignore""","""Directory"""
"""val""","""val/labels""","""Directory"""
"""val""","""val/annotations_coco.json""","""File"""


In [4]:
frames, objects, ignored_regions = load_tables(dataset_root)

pl.DataFrame({"YOLO class ID": range(8), "COCO category ID": range(1, 9), "Class": CLASS_NAMES})

2026-11-02T22:54:14 [INFO] hm2.data: Reading train COCO metadata
2026-11-02T22:54:22 [INFO] hm2.data: Reading val COCO metadata


YOLO class ID,COCO category ID,Class
i64,i64,str
0,1,"""person"""
1,2,"""bicycle"""
2,3,"""car"""
3,4,"""van"""
4,5,"""truck"""
5,6,"""tricycle"""
6,7,"""bus"""
7,8,"""motor"""


In [5]:
expected_inventory = pl.DataFrame(
    {
        "split": ["train", "val", "test"],
        "expected_frames": [24201, 2846, 3811],
        "expected_clips": [56, 7, 48],
    }
)

inventory = frames.group_by("split").agg(
    pl.len().alias("metadata_frames"),
    pl.col("clip").n_unique().alias("clips"),
    pl.col("image_width").min().alias("min_width_px"),
    pl.col("image_width").max().alias("max_width_px"),
    pl.col("image_height").min().alias("min_height_px"),
    pl.col("image_height").max().alias("max_height_px"),
)

file_inventory = []

for split in ("train", "val", "test"):
    paths = list((dataset_root / split / "images").glob("*/*.jpg"))
    file_inventory.append(
        {
            "split": split,
            "image_files": len(paths),
            "file_clips": len({path.parent.name for path in paths}),
            "has_annotations": (dataset_root / split / "annotations_coco.json").is_file(),
            "label_files": len(list((dataset_root / split / "labels").glob("*/*.txt"))),
            "ignore_files": len(list((dataset_root / split / "ignore").glob("*/*.txt"))),
        }
    )

inventory = (
    inventory.join(pl.from_dicts(file_inventory), on="split")
    .join(expected_inventory, on="split")
    .with_columns(
        (
            (pl.col("metadata_frames") == pl.col("expected_frames"))
            & (pl.col("image_files") == pl.col("expected_frames"))
            & (pl.col("clips") == pl.col("expected_clips"))
            & (pl.col("file_clips") == pl.col("expected_clips"))
        ).alias("is_complete")
    )
    .sort("split")
)

inventory

split,metadata_frames,clips,min_width_px,max_width_px,min_height_px,…,has_annotations,label_files,ignore_files,expected_frames,expected_clips,is_complete
str,u32,u32,i64,i64,i64,…,bool,i64,i64,i64,i64,bool
"""test""",3811,48,680,1920,382,…,false,0,0,3811,48,true
"""train""",24201,56,1344,2720,756,…,true,24201,18090,24201,56,true
"""val""",2846,7,1344,3840,756,…,true,2846,1434,2846,7,true


In [6]:
Markdown(
    f"**Observation.** The inventory contains {frames.height:,} image records. {inventory.filter(pl.col('is_complete')).height} of 3 splits match the published frame and clip counts. Any mismatch needs investigation before experiments."
)

**Observation.** The inventory contains 30,858 image records. 3 of 3 splits match the published frame and clip counts. Any mismatch needs investigation before experiments.

## 4. Annotation records and file integrity

COCO stores pixel top-left `x y width height` with category IDs 1 to 8. YOLO stores
normalized center `cx cy width height` with class IDs 0 to 7. Ignored-region text files
omit the class. We keep their coordinates and raw attributes without clipping boxes.

The downloaded COCO files repeat each ignored region for all eight target categories.
We group identical crowd geometry for region counts and overlays, retaining all source
annotation IDs and category IDs in list columns. This does not change the raw files.

In [7]:
test_metadata = pl.read_csv(dataset_root / "test_images.csv")

test_metadata.head(3)

image_id,file_name,clip,frame,width,height
str,str,str,i64,i64,i64
"""clip_001_0000001""","""test/images/clip_001/0000001.j…","""clip_001""",1,1920,1080
"""clip_001_0000002""","""test/images/clip_001/0000002.j…","""clip_001""",2,1920,1080
"""clip_001_0000003""","""test/images/clip_001/0000003.j…","""clip_001""",3,1920,1080


In [8]:
if (dataset_root / "sample_submission.csv").is_file():
    sample_submission = pl.read_csv(dataset_root / "sample_submission.csv")
    display(sample_submission.head(3))
else:
    display(
        Markdown(
            "**Inventory mismatch.** `sample_submission.csv` is absent from the downloaded competition files. Sample-row validation is unavailable. We have not generated a replacement or any submission predictions."
        )
    )

objects.select(
    "image_key",
    "category_id",
    "class_id",
    "class_name",
    "x",
    "y",
    "width",
    "height",
    "occlusion",
    "truncation",
    "track_id",
    "orig_category_id",
).head(5)

**Inventory mismatch.** `sample_submission.csv` is absent from the downloaded competition files. Sample-row validation is unavailable. We have not generated a replacement or any submission predictions.

image_key,category_id,class_id,class_name,x,y,width,height,occlusion,truncation,track_id,orig_category_id
str,i64,i64,str,f64,f64,f64,f64,i64,i64,i64,i64
"""train/uav0000013_00000_v/00000…",1,0,"""person""",424.0,543.0,35.0,88.0,0,0,2,1
"""train/uav0000013_00000_v/00000…",1,0,"""person""",416.0,530.0,36.0,85.0,1,0,3,1
"""train/uav0000013_00000_v/00000…",6,5,"""tricycle""",531.0,641.0,87.0,114.0,0,1,4,7
"""train/uav0000013_00000_v/00000…",1,0,"""person""",591.0,422.0,32.0,79.0,0,0,5,1
"""train/uav0000013_00000_v/00000…",1,0,"""person""",454.0,413.0,21.0,48.0,0,0,6,1


In [9]:
ignored_regions.select(
    "image_key", "x", "y", "width", "height", "coco_replication_count", "coco_category_ids"
).head(3)

image_key,x,y,width,height,coco_replication_count,coco_category_ids
str,f64,f64,f64,f64,u32,list[i64]
"""train/uav0000013_00000_v/00000…",593.0,43.0,174.0,190.0,8,"[1, 2, … 8]"
"""train/uav0000013_00000_v/00000…",489.0,307.0,41.0,62.0,8,"[1, 2, … 8]"
"""train/uav0000013_00000_v/00000…",635.0,231.0,39.0,62.0,8,"[1, 2, … 8]"


In [10]:
integrity = audit_dataset(dataset_root, frames, objects, ignored_regions)

integrity

check,count,is_pass
str,i64,bool
"""Duplicate image keys""",0,true
"""Duplicate split image IDs""",0,true
"""Duplicate annotation IDs""",0,true
"""Annotations with unknown image…",0,true
"""Missing image files""",0,true
"""Images absent from metadata""",0,true
…,…,…
"""Missing sample submission""",1,false
"""Invalid image dimensions""",0,true


In [11]:
header_checks = []

sampled_frames = frames.sample(n=min(18, frames.height), seed=SEED)

resolution_extremes = (
    frames.sort("image_width")
    .group_by("split", maintain_order=True)
    .agg(pl.col("image_key").first().alias("smallest"), pl.col("image_key").last().alias("largest"))
)

extreme_keys = resolution_extremes["smallest"].to_list() + resolution_extremes["largest"].to_list()

sampled_frames = pl.concat(
    [sampled_frames, frames.filter(pl.col("image_key").is_in(extreme_keys))]
).unique("image_key")

for record in tqdm(sampled_frames.iter_rows(named=True), total=sampled_frames.height):
    with Image.open(dataset_root / record["relative_path"]) as image:
        header_checks.append(
            {
                "image_key": record["image_key"],
                "header_width": image.width,
                "header_height": image.height,
                "is_match": image.size == (record["image_width"], record["image_height"]),
            }
        )

header_checks = pl.from_dicts(header_checks)

coordinate_checks = compare_yolo_coco(
    dataset_root, frames, objects, ignored_regions, sample_size=24, seed=SEED
)

header_checks

  0%|          | 0/24 [00:00<?, ?it/s]

image_key,header_width,header_height,is_match
str,i64,i64,bool
"""train/uav0000360_00001_v/00000…",1904,1071,true
"""val/uav0000086_00000_v/0000037""",1344,756,true
"""train/uav0000289_00001_v/00000…",1904,1071,true
"""train/uav0000264_02760_v/00000…",1904,1071,true
"""val/uav0000268_05773_v/0000008""",3840,2160,true
"""train/uav0000300_00000_v/00003…",1904,1071,true
…,…,…,…
"""train/uav0000316_01288_v/00001…",2720,1530,true
"""test/clip_041/0000120""",1920,1080,true


In [12]:
coordinate_checks.head(8)

image_key,targets,ignored_regions,max_error_px,is_match
str,i64,i64,f64,bool
"""train/uav0000266_03598_v/00001…",7,2,0.0016,true
"""train/uav0000288_00001_v/00009…",48,0,0.001136,true
"""train/uav0000309_00000_v/00000…",51,0,0.001304,true
"""train/uav0000361_02323_v/00001…",87,6,0.00132,true
"""train/uav0000020_00406_v/00003…",16,1,0.000768,true
"""val/uav0000117_02622_v/0000073""",56,3,0.00192,true
"""train/uav0000342_04692_v/00002…",53,5,0.001168,true
"""val/uav0000086_00000_v/0000104""",40,0,0.000896,true


In [13]:
Markdown(
    f"**Resolution check.** Source metadata reaches {frames['image_width'].max()} × {frames['image_height'].max()} px, above the 1920 × 1080 maximum in the supplied context. The header sample includes resolution extremes from every split."
)

**Resolution check.** Source metadata reaches 3840 × 2160 px, above the 1920 × 1080 maximum in the supplied context. The header sample includes resolution extremes from every split.

In [14]:
Markdown(
    f"**Observation.** {header_checks.filter(~pl.col('is_match')).height} sampled image headers and {coordinate_checks.filter(~pl.col('is_match')).height} sampled YOLO/COCO comparisons disagree. The comparison tolerance is 0.01 original-image pixels to allow for six-decimal YOLO rounding."
)

**Observation.** 0 sampled image headers and 0 sampled YOLO/COCO comparisons disagree. The comparison tolerance is 0.01 original-image pixels to allow for six-decimal YOLO rounding.

## 5. Save metadata for EDA

Parquet avoids repeatedly parsing the COCO JSON in the analysis notebook. The manifest
records the source metadata sizes and modification times, the resolved dataset path,
and the seed. `01_eda.ipynb` checks this manifest before loading the tables.

In [15]:
METADATA_DIRECTORY.mkdir(parents=True, exist_ok=True)

outputs = []

for name, table in (("frames", frames), ("objects", objects), ("ignored_regions", ignored_regions)):
    output_path = METADATA_DIRECTORY / f"{name}.parquet"
    table.write_parquet(output_path)
    outputs.append(
        {"Table": name, "Rows": table.height, "Size MiB": output_path.stat().st_size / 1024**2}
    )

source_names = [
    "train/annotations_coco.json",
    "val/annotations_coco.json",
    "test_images.csv",
    "sample_submission.csv",
]

source_names = [name for name in source_names if (dataset_root / name).is_file()]

manifest = {
    "dataset_root": str(dataset_root),
    "seed": SEED,
    "sources": {
        name: {
            "size": (dataset_root / name).stat().st_size,
            "mtime_ns": (dataset_root / name).stat().st_mtime_ns,
        }
        for name in source_names
    },
    "has_inventory_mismatches": not inventory["is_complete"].all(),
    "integrity_anomaly_count": int(integrity["count"].sum()),
}

manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

integrity.write_csv(METADATA_DIRECTORY / "integrity.csv")

inventory.write_csv(METADATA_DIRECTORY / "inventory.csv")

pl.from_dicts(outputs)

Table,Rows,Size MiB
str,i64,f64
"""frames""",30858,0.119818
"""objects""",1219648,9.670856
"""ignored_regions""",67838,1.278781


The outputs are `data/metadata/frames.parquet`, `objects.parquet`,
`ignored_regions.parquet`, the source manifest, and inventory checks. Raw images,
labels, and ignored regions have not been resized or changed.

Open `01_eda.ipynb` next. It requires these tables and the original images at the path
recorded in the manifest. Review any reported anomalies before using the data for training.